# Data Cleaning Fundamentals

## Why This Matters

Real-world data is messy. When you pull data from an API, download a CSV from a vendor, or query a database, you'll encounter:
- Missing values (the API timed out, the sensor failed, the field was optional)
- Duplicates (the same record was inserted twice)
- Outliers (someone entered $1500 instead of $15.00)
- Formatting issues (tickers stored as "AAPL", "aapl", and " AAPL ")

If you feed dirty data into your analysis, you get garbage results. This is especially true in quantitative finance where a single bad price can completely distort a backtest or risk calculation.

By the end of this lesson, you'll know how to:
1. Detect and handle **missing values**
2. Identify and treat **outliers**
3. Find and remove **duplicates**
4. Clean **messy text data**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

# For reproducibility
np.random.seed(42)

print("Ready to clean some data!")

---

# Part 1: Missing Values

Missing values are the most common data quality issue. They can arise from:
- **Data collection errors**: Sensor failures, API timeouts
- **Data merging**: Joining tables with different coverage
- **Historical gaps**: Markets closed, company didn't exist yet
- **Intentional omission**: Privacy, confidentiality

Python has several representations for "missing":

## The Three Faces of Missing Data

In [ ]:
# 1. Python's None
print("None:")
print(f"  Type: {type(None)}")
print(f"  Used for: General absence of value")

# 2. NumPy's NaN (Not a Number)
print("\nnp.nan:")
print(f"  Type: {type(np.nan)}")
print(f"  Value: {np.nan}")
print(f"  Used for: Missing numeric data")

# 3. NumPy's Inf (Infinity)
print("\nnp.inf:")
print(f"  Type: {type(np.inf)}")
print(f"  Value: {np.inf}")
print(f"  Used for: Results of division by zero, overflow")

### The Sneaky Behavior of NaN

NaN has some counterintuitive properties that can bite you:

In [ ]:
# NaN propagates through calculations
print("NaN is contagious:")
print(f"  1 + np.nan = {1 + np.nan}")
print(f"  np.nan * 0 = {np.nan * 0}")
print(f"  np.nan == np.nan: {np.nan == np.nan}")  # This is False!

# Array operations with NaN
arr = np.array([1, 2, np.nan, 4])
print(f"\nArray: {arr}")
print(f"  sum(): {arr.sum()}")      # NaN!
print(f"  mean(): {arr.mean()}")    # NaN!
print(f"  nansum(): {np.nansum(arr)}")   # Ignores NaN
print(f"  nanmean(): {np.nanmean(arr)}") # Ignores NaN

In [ ]:
# None causes errors in numeric operations
print("None is stricter:")
try:
    result = 1 + None
except TypeError as e:
    print(f"  1 + None raises: {e}")

# But pandas converts None to NaN in numeric contexts
s = pd.Series([1, 2, None, 4])
print(f"\nPandas converts None to NaN:")
print(s)

In [ ]:
# Infinity also causes issues
print("Infinity behavior:")
print(f"  1 + np.inf = {1 + np.inf}")
print(f"  np.inf - np.inf = {np.inf - np.inf}")  # NaN!
print(f"  np.inf / np.inf = {np.inf / np.inf}")  # NaN!

arr = np.array([1, np.inf, 3, 4])
print(f"\nArray with inf: {arr}")
print(f"  sum(): {arr.sum()}")  # Inf
print(f"  mean(): {arr.mean()}")  # Inf

## Detecting Missing Values

Let's create a realistic example with missing data:

In [ ]:
# Simulated trading data with various issues
dates = pd.date_range('2024-01-01', periods=20, freq='B')

df = pd.DataFrame({
    'date': dates,
    'price': [100, 101, np.nan, 103, 102, None, 105, 106, 104, 107,
              108, np.nan, 110, np.nan, 112, 113, 114, 115, np.nan, 117],
    'volume': [1000, 1100, 1050, np.inf, 1200, 1150, None, 1300, 1250, 1400,
               1350, 1500, 1450, 1600, 1550, np.nan, 1700, 1650, 1800, 1750],
    'sector': ['Tech', 'Tech', 'Tech', 'Finance', 'Finance', None, 'Tech', 
               'Energy', 'Energy', 'Tech', 'Finance', 'Finance', 'Tech',
               'Energy', 'Energy', 'Tech', 'Finance', None, 'Tech', 'Energy']
})

df

### Method 1: `.info()` - Quick Overview

In [ ]:
# Quick overview shows non-null counts
df.info()

**Reading the output**: Look at the "Non-Null Count" column. If it's less than 20 (our row count), we have missing values.

But this doesn't catch `np.inf`! We need more tools.

### Method 2: Detection Functions

In [ ]:
# pd.isnull() detects NaN and None (but NOT inf)
print("pd.isnull() results:")
print(pd.isnull(df['price']))

In [ ]:
# Count nulls per column
print("Null counts by column:")
print(df.isnull().sum())

In [ ]:
# np.isfinite() catches BOTH NaN and Inf (for numeric columns)
print("\nnp.isfinite() on volume:")
print(np.isfinite(df['volume']))

print(f"\nNon-finite values in volume: {(~np.isfinite(df['volume'])).sum()}")

In [ ]:
# Comprehensive missing value report
def missing_report(df):
    """Generate a comprehensive missing value report."""
    report = pd.DataFrame({
        'dtype': df.dtypes,
        'null_count': df.isnull().sum(),
        'null_pct': (df.isnull().sum() / len(df) * 100).round(1)
    })
    
    # Check for inf in numeric columns
    inf_count = []
    for col in df.columns:
        if df[col].dtype in ['float64', 'int64']:
            inf_count.append(np.isinf(df[col]).sum())
        else:
            inf_count.append(0)
    report['inf_count'] = inf_count
    
    return report

missing_report(df)

## Handling Missing Values

You have three main strategies:

1. **Drop** - Remove rows/columns with missing data
2. **Fill** - Replace with a specific value or computed value
3. **Interpolate** - Estimate based on surrounding values (great for time series)

The choice depends on **why** the data is missing and **what** you're doing with it.

### Strategy 1: Dropping Missing Values

In [ ]:
# Create a sample DataFrame
sample = pd.DataFrame({
    'A': [1, 2, np.nan, 4],
    'B': [5, np.nan, np.nan, 8],
    'C': [9, 10, 11, 12]
})
print("Original:")
print(sample)

# Drop any row with ANY null
print("\ndropna() - drop rows with any null:")
print(sample.dropna())

In [ ]:
# Drop only if ALL values in row are null
print("dropna(how='all') - drop only all-null rows:")
print(sample.dropna(how='all'))

# Drop based on specific columns
print("\ndropna(subset=['A']) - drop if A is null:")
print(sample.dropna(subset=['A']))

In [ ]:
# Drop if fewer than N non-null values
print("dropna(thresh=2) - keep rows with at least 2 non-null values:")
print(sample.dropna(thresh=2))

# Drop columns instead of rows
print("\ndropna(axis=1) - drop columns with any null:")
print(sample.dropna(axis=1))

**When to Drop**:
- Small percentage of missing values (< 5%)
- Data is missing completely at random
- You have plenty of data

**When NOT to Drop**:
- Systematic missingness (would introduce bias)
- Limited data
- Time series with gaps you need to preserve

### Strategy 2: Filling Missing Values

In [ ]:
prices = pd.Series([100, 101, np.nan, np.nan, 105, 106, np.nan, 108])
print("Original prices:")
print(prices)

# Fill with a constant
print("\nFill with 0:")
print(prices.fillna(0))

In [ ]:
# Fill with statistics
print("Fill with mean:", prices.mean().round(2))
print(prices.fillna(prices.mean()))

print("\nFill with median:", prices.median())
print(prices.fillna(prices.median()))

In [ ]:
# Forward fill (ffill) - use last known value
print("Forward fill (great for time series):")
print(prices.ffill())

# Backward fill (bfill) - use next known value
print("\nBackward fill:")
print(prices.bfill())

In [ ]:
# For DataFrames, you can fill different columns differently
df_clean = df.copy()

# Fill price with forward fill (last known price)
df_clean['price'] = df_clean['price'].ffill()

# Fill volume with median (robust to outliers)
median_vol = df_clean['volume'].replace([np.inf, -np.inf], np.nan).median()
df_clean['volume'] = df_clean['volume'].replace([np.inf, -np.inf], np.nan).fillna(median_vol)

# Fill sector with 'Unknown'
df_clean['sector'] = df_clean['sector'].fillna('Unknown')

print("After targeted filling:")
print(df_clean)

### Strategy 3: Interpolation (Advanced)

Interpolation estimates missing values based on surrounding values - perfect for time series!

In [ ]:
# Create a time series with gaps
ts = pd.Series([100, 102, np.nan, np.nan, 110, 112, np.nan, 118, 120],
               index=pd.date_range('2024-01-01', periods=9))

print("Original:")
print(ts)

# Linear interpolation
print("\nLinear interpolation:")
print(ts.interpolate(method='linear'))

In [ ]:
# Visualize different interpolation methods
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Original
axes[0, 0].plot(ts, 'o-', label='Original', markersize=10)
axes[0, 0].set_title('Original (with NaN)')

# Linear
ts_linear = ts.interpolate(method='linear')
axes[0, 1].plot(ts, 'o', markersize=10, label='Original')
axes[0, 1].plot(ts_linear, '--', label='Interpolated')
axes[0, 1].set_title('Linear Interpolation')
axes[0, 1].legend()

# Forward fill for comparison
ts_ffill = ts.ffill()
axes[1, 0].plot(ts, 'o', markersize=10, label='Original')
axes[1, 0].plot(ts_ffill, '--', label='Forward Fill')
axes[1, 0].set_title('Forward Fill (for comparison)')
axes[1, 0].legend()

# Polynomial interpolation
ts_poly = ts.interpolate(method='polynomial', order=2)
axes[1, 1].plot(ts, 'o', markersize=10, label='Original')
axes[1, 1].plot(ts_poly, '--', label='Polynomial')
axes[1, 1].set_title('Polynomial Interpolation (order=2)')
axes[1, 1].legend()

plt.tight_layout()

### Missing Values Decision Tree

```
Is data missing completely at random?
├── YES: Safe to drop or fill with statistics
└── NO: Be careful - investigate why!

Is it time series data?
├── YES: Consider ffill, interpolation
└── NO: Consider mean/median/mode fill

What percentage is missing?
├── <5%: Probably safe to drop
├── 5-20%: Fill with appropriate method
└── >20%: Investigate or drop the column
```

---

# Part 2: Outliers and Bad Values

Outliers are values that are "unusually" far from other observations. But "unusual" is subjective - a 10% daily return is unusual for the S&P 500, but normal for a meme stock.

The key question: **Is this outlier a data error or a real extreme event?**

In [ ]:
# Create data with outliers
np.random.seed(42)

# Normal returns with a few outliers injected
normal_returns = np.random.normal(0.0005, 0.02, 1000)  # Mean 0.05%, std 2%
outlier_indices = [100, 500, 750, 999]
normal_returns[outlier_indices] = [0.15, -0.12, 10.0, 0.20]  # Inject outliers (one is clearly data error)

returns = pd.Series(normal_returns, 
                    index=pd.date_range('2020-01-01', periods=1000, freq='B'),
                    name='daily_returns')

print(f"Returns summary:")
print(returns.describe())

## Detecting Outliers

In [ ]:
# Visual detection - always start here!
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Time series plot
returns.plot(ax=axes[0])
axes[0].set_title('Returns Over Time')
axes[0].set_ylabel('Daily Return')

# Box plot
returns.plot(kind='box', ax=axes[1])
axes[1].set_title('Returns Distribution')

plt.tight_layout()

**Immediately visible**: There's a massive outlier around 1000% return. This is almost certainly a data error (price entered as 10x the actual value).

In [ ]:
# Method 1: Z-score (how many standard deviations from mean)
mean = returns.mean()
std = returns.std()
z_scores = (returns - mean) / std

# Typically, |z| > 3 is considered an outlier
outliers_zscore = returns[np.abs(z_scores) > 3]
print(f"Outliers by Z-score (|z| > 3): {len(outliers_zscore)}")
print(outliers_zscore)

In [ ]:
# Method 2: IQR (Interquartile Range) - more robust to extreme outliers
Q1 = returns.quantile(0.25)
Q3 = returns.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_iqr = returns[(returns < lower_bound) | (returns > upper_bound)]
print(f"Outliers by IQR method: {len(outliers_iqr)}")
print(f"\nBounds: [{lower_bound:.4f}, {upper_bound:.4f}]")
print(f"\nOutliers:")
print(outliers_iqr.head(10))

In [ ]:
# Method 3: Percentile-based (useful for known distributions)
lower_pct = returns.quantile(0.001)  # 1st percentile
upper_pct = returns.quantile(0.999)  # 99th percentile

outliers_pct = returns[(returns < lower_pct) | (returns > upper_pct)]
print(f"Outliers beyond 1st/99th percentile: {len(outliers_pct)}")
print(f"Bounds: [{lower_pct:.4f}, {upper_pct:.4f}]")

outliers_pct

## Handling Outliers

In [ ]:
# Option 1: Remove outliers
returns_no_outliers = returns[np.abs(z_scores) <= 3]
print(f"Original length: {len(returns)}")
print(f"After removing outliers: {len(returns_no_outliers)}")

In [ ]:
# Option 2: Cap/Winsorize (clip to bounds)
returns_capped = returns.clip(lower=lower_bound, upper=upper_bound)

print("Before capping:")
print(f"  Min: {returns.min():.4f}, Max: {returns.max():.4f}")
print("After capping:")
print(f"  Min: {returns_capped.min():.4f}, Max: {returns_capped.max():.4f}")

In [ ]:
# Option 3: Replace with NaN (then fill)
returns_nan = returns.copy()
returns_nan[np.abs(z_scores) > 3] = np.nan

# Then fill with median or interpolate
returns_filled = returns_nan.fillna(returns_nan.median())

print(f"Nulls after replacement: {returns_nan.isnull().sum()}")
print(f"After filling with median: {returns_filled.isnull().sum()} nulls")

In [ ]:
# Compare the results visually
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Original
returns.plot(ax=axes[0, 0], title='Original Data')

# Removed
returns_no_outliers.plot(ax=axes[0, 1], title='Outliers Removed')

# Capped
returns_capped.plot(ax=axes[1, 0], title='Outliers Capped')

# Replaced with median
returns_filled.plot(ax=axes[1, 1], title='Outliers Replaced with Median')

plt.tight_layout()

### Outlier Handling Guidelines

| Situation | Recommended Action |
|-----------|-------------------|
| Clear data error (1000% return) | Remove or replace with NaN |
| Extreme but plausible | Investigate, then cap if needed |
| Many outliers (>5%) | Re-examine your definition of "outlier" |
| Modeling returns | Cap at percentiles (e.g., 1st/99th) |
| Risk analysis | Keep outliers! They're what you're studying |

---

# Part 3: Duplicate Values

Duplicates can be:
- **Exact duplicates**: Same values in every column
- **Key duplicates**: Same identifier but different values (conflicting data!)

Both are problematic for analysis.

In [ ]:
# Create data with duplicates
df_dups = pd.DataFrame({
    'date': ['2024-01-01', '2024-01-02', '2024-01-02', '2024-01-03', 
             '2024-01-03', '2024-01-03', '2024-01-04'],
    'ticker': ['AAPL', 'AAPL', 'AAPL', 'AAPL', 'AAPL', 'MSFT', 'AAPL'],
    'price': [150.0, 151.0, 151.0, 152.0, 153.0, 300.0, 154.0],  # Note: row 3 & 4 differ!
    'source': ['API', 'API', 'Backup', 'API', 'Backup', 'API', 'API']
})

df_dups

In [ ]:
# Detect exact duplicates (all columns match)
print("Exact duplicates:")
print(df_dups.duplicated())

In [ ]:
# Detect duplicates on specific columns (key duplicates)
print("\nDuplicates on date + ticker:")
print(df_dups.duplicated(subset=['date', 'ticker']))

In [ ]:
# Show ALL duplicated rows (including first occurrence)
print("All rows involved in duplicates:")
print(df_dups[df_dups.duplicated(subset=['date', 'ticker'], keep=False)])

## Handling Duplicates

In [ ]:
# Option 1: Keep first occurrence
df_first = df_dups.drop_duplicates(subset=['date', 'ticker'], keep='first')
print("Keep first:")
print(df_first)

In [ ]:
# Option 2: Keep last occurrence
df_last = df_dups.drop_duplicates(subset=['date', 'ticker'], keep='last')
print("Keep last:")
print(df_last)

In [ ]:
# Option 3: Remove ALL duplicates (conservative)
df_none = df_dups.drop_duplicates(subset=['date', 'ticker'], keep=False)
print("Remove all duplicates:")
print(df_none)

In [ ]:
# Option 4: Aggregate duplicates (e.g., average prices)
df_agg = df_dups.groupby(['date', 'ticker']).agg({
    'price': 'mean',
    'source': 'first'  # Just take first source
}).reset_index()

print("Aggregate duplicates (average price):")
print(df_agg)

---

# Part 4: Cleaning Text Data

Text data is often the messiest. Common issues:
- Extra whitespace
- Inconsistent capitalization
- Embedded HTML/special characters
- Inconsistent formatting

In [ ]:
# Create messy text data
df_text = pd.DataFrame({
    'raw_ticker': ['  AAPL ', 'aapl', 'AAPL', '<b>MSFT</b>', 'msft  ', 'GOOG', 'goog '],
    'raw_name': ['Apple Inc.', 'APPLE INC', '  apple inc.', 'Microsoft Corp.', 
                 'MICROSOFT CORP  ', 'Alphabet Inc.', 'alphabet inc'],
    'raw_data': ['price: 150.00', 'price:151', 'Price: 152.00', 'Price:300', 
                 'price :301', 'price: 140.00', 'price:141']
})

df_text

In [ ]:
# This is a problem - should only have 3 unique tickers!
print(f"Unique tickers: {df_text['raw_ticker'].nunique()}")
print(df_text['raw_ticker'].unique())

## String Cleaning Methods

Pandas provides a `.str` accessor with many useful methods:

In [ ]:
# Strip whitespace and convert to uppercase
df_text['clean_ticker'] = df_text['raw_ticker'].str.strip().str.upper()

# Remove HTML tags
df_text['clean_ticker'] = df_text['clean_ticker'].str.replace(r'<[^>]+>', '', regex=True)

print("Cleaned tickers:")
print(df_text[['raw_ticker', 'clean_ticker']])

In [ ]:
# Verify - now we should have 3 unique tickers
print(f"Unique clean tickers: {df_text['clean_ticker'].nunique()}")
print(df_text['clean_ticker'].unique())

In [ ]:
# Clean company names
df_text['clean_name'] = (df_text['raw_name']
                         .str.strip()
                         .str.upper()
                         .str.replace(r'\s+', ' ', regex=True))  # Multiple spaces to single

print("Cleaned names:")
print(df_text[['raw_name', 'clean_name']])

In [ ]:
# Extract numeric value from text
# Pattern: optional "price", colon, optional space, then the number
df_text['extracted_price'] = (df_text['raw_data']
                               .str.extract(r'(\d+\.?\d*)')[0]
                               .astype(float))

print("Extracted prices:")
print(df_text[['raw_data', 'extracted_price']])

### Common String Methods Reference

| Method | Purpose | Example |
|--------|---------|--------|
| `.str.strip()` | Remove leading/trailing whitespace | `'  AAPL  '` → `'AAPL'` |
| `.str.lower()` | Convert to lowercase | `'AAPL'` → `'aapl'` |
| `.str.upper()` | Convert to uppercase | `'aapl'` → `'AAPL'` |
| `.str.replace()` | Replace substring/pattern | Remove HTML tags |
| `.str.extract()` | Extract with regex capture groups | Pull numbers from text |
| `.str.split()` | Split into multiple columns | `'A_B'` → `['A', 'B']` |
| `.str.contains()` | Check if contains pattern | Filter rows |
| `.str.len()` | Get string length | Validation |

---

# Summary: Your Data Cleaning Checklist

Before any analysis, run through this checklist:

## 1. Initial Inspection
```python
df.info()          # Types and non-null counts
df.describe()      # Summary statistics
df.head(), df.tail()  # First/last rows
```

## 2. Missing Values
```python
df.isnull().sum()  # Count by column
df.dropna()        # Remove
df.fillna(value)   # Fill with value
df.ffill()         # Forward fill
df.interpolate()   # Interpolate
```

## 3. Duplicates
```python
df.duplicated().sum()  # Count
df.drop_duplicates()   # Remove
```

## 4. Outliers
```python
# Detect
z_scores = (df['col'] - df['col'].mean()) / df['col'].std()
outliers = df[np.abs(z_scores) > 3]

# Handle
df['col'].clip(lower, upper)  # Cap
df.loc[condition, 'col'] = np.nan  # Replace
```

## 5. Text Cleaning
```python
df['col'].str.strip().str.upper()  # Clean whitespace and case
df['col'].str.replace(pattern, replacement)  # Replace patterns
df['col'].str.extract(regex)  # Extract values
```

## 6. Validation
- Re-run `.info()` and `.describe()` after cleaning
- Visualize key columns to verify
- Check that counts and ranges make sense

---

# What's Next?

Now you know the fundamentals. In the next notebook, we'll put it all together with a hands-on case study:

- Start with messy, realistic financial data
- Use visualization to detect problems
- Apply systematic cleaning
- Compare before/after
- Document your decisions

See you in `3_real_world_data_challenge.ipynb`!